# Notebook 02 — Published benchmarks

**Workshop:** Agentic AI for Scientists — Week 5 (Evaluation & Benchmarking)
**Block:** standardized benchmarks (20 min)
**Goal:** Your own test set answers "is it good *for my task*?" **Benchmarks** answer "how does it stack up against the field on a *standard* task?" We run a real medical-QA benchmark (**PubMedQA**) end to end, then map the **agent-benchmark** landscape you'll hear about (SWE-bench, GAIA, AgentBench, tau-bench) and where each one fits.

**Honesty up front:** a 0.6B model will *not* top PubMedQA. That is not the point. The point is to run a benchmark *correctly* — the harness, the metric, the leakage traps — so you can read benchmark claims critically and build your own.

## 0. Setup

In [ ]:
%pip install -q unsloth datasets
%pip install -q --no-deps transformers

---
## 1. PubMedQA — a real medical QA benchmark

**PubMedQA** asks a yes/no/maybe question against a PubMed abstract. We use the `pqa_labeled` split (1,000 expert-annotated items). It's small, clean, and has a single accuracy metric — ideal for a teaching run.

In [ ]:
from datasets import load_dataset

pqa = load_dataset("pubmed_qa", "pqa_labeled", split="train")
print(pqa)
ex = pqa[0]
print("\nQUESTION :", ex["question"])
print("DECISION :", ex["final_decision"])   # yes / no / maybe
print("CONTEXT  :", " ".join(ex["context"]["contexts"])[:300], "...")

In [ ]:
from unsloth import FastLanguageModel
from pathlib import Path
import torch, re

CANDIDATES = ["qwen3-medquad-qlora-merged", "qwen3-medquad-full-sft", "Qwen/Qwen3-0.6B"]
MODEL_PATH = next((c for c in CANDIDATES if c.startswith("Qwen/") or Path(c).exists()), "Qwen/Qwen3-0.6B")
print(f"Benchmarking: {MODEL_PATH}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, max_seq_length=2048, dtype=None, load_in_4bit=False)
FastLanguageModel.for_inference(model)

def answer_yesno(question, context):
    prompt = (f"Answer the question with exactly one word: yes, no, or maybe.\n\n"
              f"Context: {context[:1500]}\n\nQuestion: {question}\n\nAnswer:")
    msgs = [{"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=8, temperature=0.0, do_sample=False)
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).lower()
    for label in ("yes", "no", "maybe"):
        if re.search(rf"\b{label}\b", text):
            return label
    return "maybe"  # default for unparseable

---
## 2. Run the benchmark (small slice)

We score a **subset** (e.g. 50) so it finishes in class — and we say so out loud. *Quietly evaluating on a subset and reporting it as "PubMedQA accuracy" is exactly the kind of benchmark dishonesty you should learn to spot.* Always report N.

In [ ]:
N = 50  # subset for an in-class run. Reporting honestly: this is PubMedQA[:50], not the full 1,000.
correct = 0
from collections import Counter
preds = Counter()
for ex in pqa.select(range(N)):
    ctx = " ".join(ex["context"]["contexts"])
    pred = answer_yesno(ex["question"], ctx)
    preds[pred] += 1
    correct += (pred == ex["final_decision"])
acc = correct / N
print(f"PubMedQA[:{N}] accuracy: {acc:.1%}  ({correct}/{N})")
print(f"prediction distribution: {dict(preds)}")
print(f"(majority-class baseline = always 'yes': {sum(1 for e in pqa.select(range(N)) if e['final_decision']=='yes')/N:.1%})")

> **Read it critically.** Compare accuracy to the *majority-class baseline*. A model that always answers "yes" can look ~55% accurate on PubMedQA because the labels are skewed. Beating the majority baseline — not the raw percentage — is what tells you the model knows anything. This single habit catches a huge fraction of misleading benchmark claims.

---
## 3. The agent-benchmark landscape

Single-turn QA benchmarks (PubMedQA, MMLU) measure *knowledge*. **Agentic** systems need benchmarks that measure *multi-step action* — using tools, browsing, writing code, recovering from errors. The ones you'll hear cited:

| Benchmark | Measures | Task shape | Why it's hard |
|---|---|---|---|
| **SWE-bench** | real software engineering | fix a real GitHub issue, tests must pass | long-horizon, real repos, verifiable |
| **GAIA** | general assistant reasoning | multi-step questions needing tools/web | needs planning + tool use, not recall |
| **AgentBench** | agent ability across 8 envs | OS, DB, web, games | breadth; many distinct action spaces |
| **tau-bench** | tool-agent reliability | customer-service tool calls vs a policy | consistency across runs, rule-following |
| **MedAgentBench** | clinical agent tasks | EHR actions, order entry | domain + safety + multi-step |

**How to read an agent-benchmark number:**
- **Verifiable > judged.** SWE-bench (tests pass / fail) is harder to game than a judged score.
- **pass@1 vs pass@k.** "Solved 40%" at pass@8 (eight tries) is very different from pass@1.
- **Contamination.** If the benchmark predates the model's training cutoff, the model may have *seen* it. Always check dates.
- **Your task != the benchmark.** A high GAIA score doesn't mean the agent is good at *your* lab's workflow. Which is exactly why you also build your own eval (next notebook).

## What you learned

You ran a real benchmark honestly (reported N, compared to a baseline) and learned to read agent-benchmark claims with the right suspicion: verifiable beats judged, pass@1 beats pass@k, check for contamination, and a leaderboard score is never a substitute for evaluating on *your* task.

**Next — `03_custom_eval_harness.ipynb`:** build a reusable, assertion-based regression suite for your own agent — so "did my change make it better or worse?" becomes a one-command answer.